# Daedalus Devs Team Report # 1 (03/24/2026)

### What we have done this week:
* Installed libraries and dependencies needed **(cell 1)**
* Figured out how to upload resume PDF file to Colab and use it with the LLM
* Created a function that takes in user's resume and parse it to output the extracted info **(cell 3)**
* Created a function that uses Skyvern, played around a little to see how it works. **(cell 4)**

### Plans for next week:
* Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* Follow our workflow where the user can make corrections to the extracted resume info.

In [1]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Done installing


In [2]:
import os
from dotenv import load_dotenv


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

Key check: FOUND


In [2]:
url = "https://demo.applitools.com/"

In [6]:
# !skyvern quickstart
# from skyvern import Skyvern
# import asyncio

# async def run_locally():
#   client = Skyvern.local()
#   browser = await client.launch_local_browser()
#   page = await browser.get_working_page()
#   await page.goto(url)
#   await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
#   print("page filled in!\n")

#   await browser.close()

# await run_locally()

from skyvern import Skyvern
import asyncio
import nest_asyncio
from google import genai
# from google.colab import userdata

nest_asyncio.apply()

client_genai = genai.Client(api_key=gemini_key)

# this takes in user's resume and parse and LLM will output the info it extracted to ensure accuracy
def parse_pdf_with_gemini(file_path, prompt="Please parse this resume and return its contents in a structured JSON that is easy for both LLM and humans to read."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    """
    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

   # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        # model="gemini-2.0-flash",
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    return response.text

# check if agent extracted the resume's info correctly
def verify_resume_info(extracted_info):
    """
    Shows the extracted info to the user and allows them to correct it.
    Keeps looping until the user confirms it's correct.
    """
    current_info = extracted_info

    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        print(current_info)
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            return current_info
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}

The user wants the following correction or addition:
{correction}

Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")


# call the function to store the extracted info
result = parse_pdf_with_gemini("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

print("\nFinal verified resume info:")
print(final_info)

# # call the function to store the extracted info and print it
# result = parse_pdf_with_gemini("content/jakes-resume.pdf")
# print(result)




Uploading content/jakes-resume.pdf...

EXTRACTED RESUME INFO:
```json
{
  "name": "Jake Ryan",
  "contact": {
    "phone": "123-456-7890",
    "email": "jake@su.edu",
    "linkedin": "linkedin.com/in/jake",
    "github": "github.com/jake"
  },
  "education": [
    {
      "institution": "Southwestern University",
      "degree": "Bachelor of Arts in Computer Science",
      "minor": "Business",
      "location": "Georgetown, TX",
      "dates": "Aug. 2018 - May 2021"
    },
    {
      "institution": "Blinn College",
      "degree": "Associate's in Liberal Arts",
      "location": "Bryan, TX",
      "dates": "Aug. 2014 - May 2018"
    }
  ],
  "experience": [
    {
      "title": "Undergraduate Research Assistant",
      "institution": "Texas A&M University",
      "location": "College Station, TX",
      "dates": "June 2020 - Present",
      "description": [
        "Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems",
        "Developed a

In [ ]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

CHROMA_HOST: api.trychroma.com
CHROMA_TENANT: 7eab8b90-a03c-4bc3-9704-e5e4f8a3f82b
CHROMA_DATABASE: phil-job-agent
CHROMA_API_KEY: FOUND
Connected to ChromaDB! Total resumes stored: 0


NameError: name 'hashlib' is not defined

In [ ]:
from skyvern import Skyvern

async def run_locally():
    client = Skyvern.local()
    browser_args = [
        "--disable-gpu",
        "--disable-software-rasterizer",
        "--no-sandbox"
    ]
    browser = await client.launch_local_browser(headless=False, args=browser_args)  # Make browser visible
    page = await browser.get_working_page()
    await page.goto("https://demo.applitools.com/")
    print("Done opening page!\n")
    await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
    print("Page filled in!\n")
    # Note: Browser will remain open; close manually if needed
    # await browser.close()  # Uncomment to close after

await run_locally()



# import os
# from dotenv import load_dotenv
# from skyvern import Skyvern
# import nest_asyncio

# nest_asyncio.apply()
# load_dotenv()

# async def run_locally():
#     skyvern = Skyvern(api_key=os.getenv("SKYVERN_API_KEY"))
    
#     browser = await skyvern.launch_local_browser(headless=False, args=[
#             "--disable-gpu",
#             "--disable-software-rasterizer",
#             "--no-sandbox",
#             "--disable-dev-shm-usage",  # important for WSL
#         ])
#     page = await browser.get_working_page()
#     await page.goto("https://demo.applitools.com/")
#     print("Done opening page!\n")
#     await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
#     print("Page filled in!\n")

# await run_locally()

2026-03-29T00:19:02.730482Z [info     ] [skyvern_browser_page_ai.py    :214 ] AI act                         [skyvern.library.skyvern_browser_page_ai] prompt=My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login workflow_run_id=None entrypoint=ipykernel_launcher


Done opening page!



/Users/sanilachowdhury/Desktop/ai_agents/job-agent/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:365: InsecureKeyLengthWarning: The HMAC key is 11 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  decoded = self.decode_complete(
2026-03-29T00:19:05.714726Z [info     ] [sdk.py                        :47  ] Running SDK action             [skyvern.forge.sdk.routes.sdk] organization_id=o_511343802250716860 action_type=ai_act entrypoint=ipykernel_launcher
2026-03-29T00:19:05.717655Z [info     ] [service.py                    :4389] Creating workflow from request [skyvern.forge.sdk.workflow.service] organization_id=o_511343802250716860 title=SDK Workflow entrypoint=ipykernel_launcher
2026-03-29T00:19:06.083947Z [info     ] [workflow_definition_converter.py:326 ] Created workflow from request  [skyvern.forge.sdk.workflow.workflow_definition_converter] parameter_keys=[] block_labels=[] workflow_id=w_511360290415767754 entrypoint=i

ApiError: headers: {'content-length': '29', 'content-type': 'application/json'}, status_code: 400, body: {'detail': 'Scraping failed.'}

# Plans for next week:
- Need to figure out how to see the filled in website, not as a screenshot but in another tab.
- Follow our workflow where the user can make corrections to the extracted resume info.
- put agent graph at the top of notebook for next week